# PathSnap — Notebook 02: Data processing, labels, candidates and features
**Milestone 3 · Owners: A (kumarsaavan07-jpg: labels, candidates) + B (GeniePlunder: features)**

**Partitions:** `train` (80 trips) for every decision; `student_test` (10 trips) is processed with the *same*
code and only reported as a check. Evaluator trips are never touched.

**What this notebook produces**
1. Per-GPS-point **reference labels** derived from the ordered `.route` (heading- and order-aware alignment, D6/D9).
2. Per-point **candidate arcs** with the approved rule: arcs within min(d_nearest + 20 m, 150 m) (D7, D11).
3. A **feature table**: one row per (GPS point, candidate arc), 16 features + label → input for every M4 model.
4. Evidence for every choice: label-threshold distribution, alignment sensitivity, naive-vs-aligned labels,
   feature separation (train only), rule-based baselines, Folium maps of the labels.

All logic lives in the shared package `pathsnap/` so the API (M9) runs **exactly** the same preprocessing.
Nothing here is *fitted*: features are deterministic functions of the GPS track + road network, so there is no
preprocessing leakage. Scaling (where a model needs it) is fitted inside each model's pipeline on train only (M4).

In [ ]:
# ---- 1. Setup ---------------------------------------------------------------------------------
import os, sys, zipfile, time, json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/PathSnap"
    ZIP_PATH = f"{DRIVE}/Student folder.zip"
    EXTRACT_DIR = "/content/pathsnap_data"
    OUT_DIR = f"{DRIVE}/m3_outputs"
    # the pathsnap package: from the GitHub repo if it is there, otherwise from Drive (MyDrive/PathSnap/pathsnap/)
    REPO = "/content/01_pathsnap"
    if not os.path.isdir(REPO):
        os.system(f"git clone -q https://github.com/paya5am/01_pathsnap {REPO}")
    PKG_PARENT = REPO if os.path.isdir(f"{REPO}/pathsnap") else DRIVE
else:
    ZIP_PATH = os.environ.get("PATHSNAP_ZIP", "Student folder.zip")
    EXTRACT_DIR = os.environ.get("PATHSNAP_EXTRACT", "pathsnap_data")
    OUT_DIR = os.environ.get("PATHSNAP_OUT", "m3_outputs")
    PKG_PARENT = os.environ.get("PATHSNAP_PKG", ".")
sys.path.insert(0, PKG_PARENT)
import pathsnap
from pathsnap import config as C
from pathsnap.io import read_trip, list_trips, trip_dir
from pathsnap.geo import make_projection, seg_project
from pathsnap.network import build_network
from pathsnap.track import motion_features
from pathsnap.candidates import generate_candidates
from pathsnap.labels import align_route, STATUS
from pathsnap.features import FEATURES
from pathsnap.pipeline import process_trip
print("pathsnap", pathsnap.__version__, "from", PKG_PARENT)

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
DATA_ROOT = next(dp for dp, dn, _ in os.walk(EXTRACT_DIR) if "train" in dn and "student_test" in dn)
for sub in ["tables", "figures", "maps", "dataset"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
TAB, FIG, MAPS, DS = (os.path.join(OUT_DIR, s) for s in ["tables", "figures", "maps", "dataset"])
try:
    display
except NameError:
    display = print
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("DATA_ROOT:", DATA_ROOT, "| OUT_DIR:", OUT_DIR)
print({k: getattr(C, k) for k in dir(C) if k.isupper()})

## 2. The processing pipeline (what `pathsnap.pipeline.process_trip` does, step by step)
| Step | Module | What happens | Why (evidence) |
|---|---|---|---|
| 1 | `io.read_trip` | read `.track/.nodes/.arcs/.route` (no headers, whitespace) | formats verified in 01 §2–3, V1–V2 |
| 2 | `geo.make_projection` | lon/lat → metres, per-trip equirectangular centred on the track mean | max error 0.72 m (01 §4) |
| 3 | `network.build_network` | arc lengths & bearings; **drop zero-length arcs**; **merge exact duplicate (u,v) arcs** to the lowest id; two-way flag; node out/in-degree; KD-tree of arcs densified every 5 m near the track | 106 zero-length / 1,903 duplicate arcs (01 V4, §6) |
| 4 | `track.motion_features` | per point: heading & speed from positions ±2 s (4-s baseline), "moving" if ≥ 3 m/s and not spanning a gap, turn angle | 1-s GPS jitter makes 1-step headings noisy (01 §5) |
| 5 | `candidates.generate_candidates` | arcs within **min(d_nearest + 20 m, 150 m)** | D7, 01 §10 |
| 6 | `features.candidate_features` | 16 features per (point, candidate) — §6 below | |
| 7 | `labels.align_route` (train/dev only) | monotone DP alignment of points to route positions with heading penalty → reference arc per point + status | D6, D9; §4 below |

The API (M9) runs steps 1–6 only — it never needs the route.

In [ ]:
# ---- 3. Build the dataset for all trips ---------------------------------------------------------
t0 = time.time()
F_list, P_list, T_list = [], [], []
for part in ["train", "student_test"]:
    for tid in list_trips(DATA_ROOT, part):
        s = time.time()
        f, p, _ = process_trip(trip_dir(DATA_ROOT, part, tid))
        f.insert(1, "partition", part); p.insert(1, "partition", part)
        F_list.append(f); P_list.append(p)
        T_list.append({"partition": part, "trip_id": tid, "points": len(p), "rows": len(f), "seconds": time.time() - s})
FEAT = pd.concat(F_list, ignore_index=True); PTS = pd.concat(P_list, ignore_index=True); TIM = pd.DataFrame(T_list)
TIM.to_csv(f"{TAB}/01_processing_time.csv", index=False)
print(f"{len(TIM)} trips, {len(PTS):,} GPS points, {len(FEAT):,} candidate rows in {time.time()-t0:.0f}s")
print("processing time per trip: median %.2fs, max %.2fs (trip %d)" % (TIM.seconds.median(), TIM.seconds.max(), TIM.loc[TIM.seconds.idxmax(), "trip_id"]))

## 4. Reference labels: how they are derived and how trustworthy they are
`.route` is an ordered list of arcs (median 5.25 GPS points per arc), so per-point labels must be derived.

**Alignment (labels.align_route).** Each GPS point i is matched to a route position j(i), with j **never going
backwards in time** (order-aware). The matching minimises
`Σ_i  min(dist(i, arc j), 200 m) + 100 m × [i is moving and arc j points against the motion]`
by dynamic programming (O(points × route arcs), like DTW). The heading term is what makes out-and-back trips
(01 V5, trip 11) come out right; the 200 m cap stops parked clouds dominating the alignment.

**Status after alignment.** A point is *labelled* unless (a) its matched arc is > 50 m away (`unlabelled_far`:
parked off-network, past the route's end), or (b) it sits in a **sustained run (≥ 10 points)** of matches
pointing against the motion (`unlabelled_heading`: local reference error, e.g. trip 66). Single-point heading
mismatches are kept: they are heading noise at sharp turns (checked below). Unlabelled points are never used to
train or score a model, and their count is always reported.

In [ ]:
print("Label status (share of GPS points):")
display(pd.crosstab(PTS.partition, PTS.label_status.map(STATUS), normalize="index").round(4))
display(pd.crosstab(PTS.partition, PTS.label_status.map(STATUS)))
unl = PTS[PTS.label_status != 0]
per_trip_unl = unl.groupby(["partition", "trip_id"]).size().sort_values(ascending=False)
per_trip_unl.to_csv(f"{TAB}/02_unlabelled_per_trip.csv")
print("Unlabelled points per trip (top 10):", per_trip_unl.head(10).to_dict())
print("Share of unlabelled points coming from trips 54/33/2 (flagged in M2):", round(unl.trip_id.isin([54, 33, 2]).mean(), 3))

# 4a. Why 50 m?  Distribution of the distance between each point and its aligned reference arc (train)
trd = PTS[PTS.partition == "train"].ref_dist
clean = PTS[(PTS.partition == "train") & ~PTS.trip_id.isin([54, 33, 2])].ref_dist
thr = pd.DataFrame([{"threshold_m": th, "share_above_all_train": (trd > th).mean(), "share_above_clean_train": (clean > th).mean()}
                    for th in (20, 30, 40, 50, 75, 100, 150)])
thr.to_csv(f"{TAB}/03_label_threshold.csv", index=False); display(thr.round(4))
h, e = np.histogram(trd.clip(0, 300), bins=np.arange(0, 305, 5))
print("points per 5-m bin (0–150 m):", h[:30].tolist())

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(e[:-1], h, width=5, align="edge", color="#1f77b4"); ax.set_yscale("log")
# who is in the far tail? (data-driven caption)
far = PTS[(PTS.partition == "train") & (PTS.ref_dist > 100) & (PTS.ref_dist <= 250)].trip_id.value_counts(normalize=True)
print("points 100–250 m from their reference arc come from trips:", far.head(3).round(3).to_dict())
ax.axvline(C.LABEL_MAX_DIST_M, c="r", ls="--", label=f"label threshold {C.LABEL_MAX_DIST_M:.0f} m")
ax.set(xlabel="distance GPS point → aligned reference arc (m)", ylabel="points (log)",
       title=f"Train: noise peak decays to ~45–55 m; 100–250 m bumps = parked trips ({', '.join(str(t) for t in far.head(2).index)}); last bar = >300 m"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG}/01_label_distance_hist.png", dpi=130); plt.show()

In [ ]:
# 4b. Single-point heading mismatches are turn noise, not reference errors
# recompute the raw "matched arc points against motion" flag for all points (status 2 only keeps sustained runs)
mis_rows = []
for part in ["train", "student_test"]:
    for tid in list_trips(DATA_ROOT, part):
        raw = read_trip(trip_dir(DATA_ROOT, part, tid)); tr = raw.track
        lon0, lat0 = tr.lon.mean(), tr.lat.mean(); P = make_projection(lon0, lat0)(tr.lon.values, tr.lat.values)
        net = build_network(raw.nodes, raw.arcs, lon0, lat0, aoi_xy=P); mo = motion_features(P, tr.t.values)
        lab = align_route(net, P, mo, raw.route, heading_run_min=1)      # run_min = 1 → flags every single mismatch
        bad = np.where(lab["label_status"] == 2)[0]
        if len(bad):
            for r in np.split(bad, np.where(np.diff(bad) > 1)[0] + 1):
                mis_rows.append({"partition": part, "trip_id": tid, "run_len": len(r),
                                 "median_turn_deg": float(np.median(mo["turn"][r])), "median_speed": float(np.median(mo["speed"][r]))})
MIS = pd.DataFrame(mis_rows); MIS.to_csv(f"{TAB}/04_heading_mismatch_runs.csv", index=False)
print("Heading-mismatch runs:", len(MIS), "| run-length counts:", MIS.run_len.value_counts().sort_index().head(10).to_dict())
print("Runs of length 1: median turn %.0f°, median speed %.1f m/s (all labelled points: median turn %.0f°)" % (
      MIS[MIS.run_len == 1].median_turn_deg.median(), MIS[MIS.run_len == 1].median_speed.median(),
      FEAT.drop_duplicates(["partition", "trip_id", "pt"]).turn.median()))
print("Runs ≥ %d points (→ unlabelled_heading):" % C.HEADING_RUN_MIN, MIS[MIS.run_len >= C.HEADING_RUN_MIN][["trip_id", "run_len"]].values.tolist())

### 4c. Are the labels robust to the alignment settings?  And are they better than the naive way?
We re-run the alignment on all **train** trips with each setting changed (cap 100/400 m, heading penalty 50/200 m,
label threshold 30/75 m, run length 5/20) and measure how many labelled points get the **same reference arc**.
We also compare with **naive labels** = nearest route arc ignoring direction and order.

In [ ]:
VARIANTS = {"base": {}, "cap 100 m": {"dist_cap": 100}, "cap 400 m": {"dist_cap": 400},
            "heading penalty 50 m": {"heading_penalty": 50}, "heading penalty 200 m": {"heading_penalty": 200},
            "label threshold 30 m": {"label_max_dist": 30}, "label threshold 75 m": {"label_max_dist": 75},
            "run length 5": {"heading_run_min": 5}, "run length 20": {"heading_run_min": 20}}
sens, naive = [], []
for tid in list_trips(DATA_ROOT, "train"):
    raw = read_trip(trip_dir(DATA_ROOT, "train", tid)); tr = raw.track
    lon0, lat0 = tr.lon.mean(), tr.lat.mean(); P = make_projection(lon0, lat0)(tr.lon.values, tr.lat.values)
    net = build_network(raw.nodes, raw.arcs, lon0, lat0, aoi_xy=P); mo = motion_features(P, tr.t.values)
    res = {k: align_route(net, P, mo, raw.route, **v) for k, v in VARIANTS.items()}
    b = res["base"]; bl = b["label_status"] == 0
    for k, r in res.items():
        rl = r["label_status"] == 0; both = bl & rl
        sens.append({"variant": k, "n": len(P), "labelled": rl.sum(), "both": both.sum(),
                     "same_arc": (r["ref_arc"][both] == b["ref_arc"][both]).sum()})
    A, B = net.nxy[net.arcs[raw.route, 0]], net.nxy[net.arcs[raw.route, 1]]
    nv = np.empty(len(P), np.int64)
    for s in range(0, len(P), 1500):
        d, _ = seg_project(P[s:s + 1500, None], A[None], B[None]); nv[s:s + 1500] = net.canonical[raw.route[d.argmin(1)]]
    rset = set(map(tuple, raw.arcs[raw.route]))
    naive.append({"trip_id": tid, "labelled": bl.sum(), "naive_differs": (nv[bl] != b["ref_arc"][bl]).sum(),
                  "route_arcs_with_reverse_in_route": int(sum((v, u) in rset for u, v in raw.arcs[raw.route]))})
SENS = pd.DataFrame(sens).groupby("variant", sort=False)[["n", "labelled", "both", "same_arc"]].sum()
SENS["labelled_share"] = SENS.labelled / SENS.n; SENS["same_arc_as_base"] = SENS.same_arc / SENS.both
SENS.to_csv(f"{TAB}/05_label_sensitivity.csv"); display(SENS[["labelled_share", "same_arc_as_base"]].round(4))
NAIVE = pd.DataFrame(naive); NAIVE["naive_differs_share"] = NAIVE.naive_differs / NAIVE.labelled
NAIVE.to_csv(f"{TAB}/06_naive_vs_aligned_labels.csv", index=False)
print("Naive labels disagree with aligned labels on %.2f%% of labelled train points" % (100 * NAIVE.naive_differs.sum() / NAIVE.labelled.sum()))
display(NAIVE.sort_values("naive_differs_share", ascending=False).head(6).round(3))

## 5. Candidate sets on labelled points
**Candidate miss rate** = share of labelled points whose reference arc is *not* among the candidates
(the guideline metric; it caps the best possible Top-1 accuracy).

In [ ]:
LAB = PTS[PTS.label_status == 0]
cm = LAB.groupby("partition").agg(labelled_points=("pt", "size"), candidate_miss_rate=("ref_in_cands", lambda s: 1 - s.mean()),
                                  mean_candidates=("n_cands", "mean"), max_candidates=("n_cands", "max"))
cm.to_csv(f"{TAB}/07_candidate_miss_rate.csv"); display(cm.round(4))
FL = FEAT[FEAT.label >= 0]
cb = FL.groupby("partition").label.agg(rows="size", positives="sum", positive_share="mean")
cb.to_csv(f"{TAB}/08_class_balance.csv"); display(cb.round(4))

## 6. Features — what they are and why (selection uses TRAIN rows only)
| Feature | Meaning | Why it should help (EDA evidence) |
|---|---|---|
| `dist` | point-to-segment distance (m) | GPS noise σ ≈ 7.5 m: the true road is usually close |
| `dist_rel` | dist − distance to the nearest candidate | parallel roads: what matters is *how much farther* than the closest |
| `dist_ratio` | (dist+1)/(d_nearest+1) | scale-free version of dist_rel |
| `dist_rank` | 0 = nearest candidate, 1 = 2nd … | ordinal view of distance |
| `n_cands` | candidates at this point | dense junctions are harder |
| `d_nearest` | distance to the closest arc | local GPS-quality proxy (large → drifted fix) |
| `cos_heading` | cos(vehicle heading − arc bearing); 0 if unknown | 95% of arcs are two-way twins (01 §6): only direction separates them |
| `heading_known` | 1 if moving ≥ 3 m/s | tells the model when `cos_heading` is meaningful |
| `speed` | m/s over 4 s, clipped at 60 | fast → motorway-like long arcs; trips 1/13/32 clipped |
| `turn` | heading change over ±4 s (deg) | junction manoeuvres |
| `arc_len_log` | log(1 + arc length) | arcs are straight segments 0–9.3 km (01 §6) |
| `dist_to_end_m` | distance of the projection from the nearer arc end | adjacent arcs touch the point only at their end (projection clamps to the junction) |
| `two_way` | reverse arc also exists | |
| `from_out_deg`, `to_out_deg` | arcs leaving the start / end node | junction complexity |
| `win_dist_mean` | mean distance of points i−1, i, i+1 (±1 s, gap-aware) to this arc | smooths single-fix noise |

Dropped after the analysis below: `heading_diff` (duplicate of cos_heading), `win_dist_max` (r = 0.91),
`win_n`, `proj_t`, `dt_prev`, `dt_next` (separation ≈ 0.50 alone; time gaps are used by the HMM in M7).
The window length (±1 s) was chosen by comparing ±1/2/3/5 s on train (cell below).

In [ ]:
from sklearn.metrics import roc_auc_score
TR = FL[FL.partition == "train"]
auc = []
for f in FEATURES:
    a = roc_auc_score(TR.label, TR[f])
    auc.append({"feature": f, "separation_auc": max(a, 1 - a), "direction": "higher → correct" if a > .5 else "lower → correct",
                "mean_correct": TR.loc[TR.label == 1, f].mean(), "mean_wrong": TR.loc[TR.label == 0, f].mean()})
AUC = pd.DataFrame(auc).sort_values("separation_auc", ascending=False); AUC.to_csv(f"{TAB}/09_feature_separation_train.csv", index=False)
display(AUC.round(3))
corr = TR[FEATURES].corr(); corr.to_csv(f"{TAB}/10_feature_correlation_train.csv")
print("Feature pairs with |r| > 0.85:", [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:] if abs(corr.loc[a, b]) > .85])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].barh(AUC.feature[::-1], AUC.separation_auc[::-1], color="#1f77b4"); ax[0].axvline(.5, c="k", lw=.8)
ax[0].set(xlim=(.45, .9), xlabel="single-feature AUC (0.5 = no signal)", title="Train: how well each feature alone separates correct vs wrong")
im = ax[1].imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1); ax[1].set_xticks(range(len(FEATURES))); ax[1].set_yticks(range(len(FEATURES)))
ax[1].set_xticklabels(FEATURES, rotation=90, fontsize=7); ax[1].set_yticklabels(FEATURES, fontsize=7); ax[1].set_title("Feature correlation (train)")
plt.colorbar(im, ax=ax[1], shrink=.8); plt.tight_layout(); plt.savefig(f"{FIG}/02_feature_separation_correlation.png", dpi=130); plt.show()

fig, axs = plt.subplots(2, 4, figsize=(16, 7))
for ax, f in zip(axs.ravel(), AUC.feature.values[:8]):
    lo, hi = TR[f].quantile([.01, .99])
    for lab_, col, nm in [(0, "#d62728", "wrong candidate"), (1, "#2ca02c", "correct arc")]:
        ax.hist(TR.loc[TR.label == lab_, f].clip(lo, hi), bins=40, density=True, alpha=.55, color=col, label=nm)
    ax.set_title(f, fontsize=9)
axs[0, 0].legend(fontsize=8); fig.suptitle("Train: distribution of the 8 strongest features, correct vs wrong candidates")
plt.tight_layout(); plt.savefig(f"{FIG}/03_feature_distributions.png", dpi=130); plt.show()

In [ ]:
# 6b. Window length for win_dist_mean: compare ±1, 2, 3, 5 s (train, labelled rows)
from pathsnap.candidates import generate_candidates
wrows = []
for tid in list_trips(DATA_ROOT, "train"):
    raw = read_trip(trip_dir(DATA_ROOT, "train", tid)); tr = raw.track; t = tr.t.values
    lon0, lat0 = tr.lon.mean(), tr.lat.mean(); P = make_projection(lon0, lat0)(tr.lon.values, tr.lat.values)
    net = build_network(raw.nodes, raw.arcs, lon0, lat0, aoi_xy=P); mo = motion_features(P, t)
    cd, dn = generate_candidates(net, P); lab = align_route(net, P, mo, raw.route)
    cd = cd[lab["label_status"][cd.pt.values] == 0].reset_index(drop=True)
    y = (cd.arc.values == lab["ref_arc"][cd.pt.values]).astype(int)
    A, B = net.nxy[net.arcs[cd.arc.values, 0]], net.nxy[net.arcs[cd.arc.values, 1]]; pt = cd.pt.values
    row = {"y": y, "dist": cd.dist.values}
    for W in (1, 2, 3, 5):
        s_ = np.zeros(len(cd)); n_ = np.zeros(len(cd))
        for k in range(-W - 1, W + 2):
            if k == 0: continue
            q = np.clip(pt + k, 0, len(P) - 1); m = (pt + k >= 0) & (pt + k < len(P)) & (np.abs(t[q] - t[pt]) <= W + 0.05)
            d, _ = seg_project(P[q[m]], A[m], B[m]); s_[m] += d; n_[m] += 1
        row[f"window ±{W} s"] = (s_ + cd.dist.values) / (n_ + 1)
    wrows.append(pd.DataFrame(row))
WD = pd.concat(wrows, ignore_index=True)
WIN = pd.DataFrame([{"variant": c, "separation_auc": max(roc_auc_score(WD.y, WD[c]), 1 - roc_auc_score(WD.y, WD[c]))}
                    for c in WD.columns if c != "y"])
WIN.to_csv(f"{TAB}/11_window_length.csv", index=False); display(WIN.round(3))
print("→ ±1 s is best; wider windows reach past the end of short arcs (median arc 25 m ≈ 2–3 s of driving).")

## 7. Rule-based baselines (point-level Top-1 on labelled points; a candidate miss counts as wrong)
These set the bar the M4 models must beat, and show why heading matters.

In [ ]:
def top1(df, score, part):
    d = df[df.partition == part].copy(); d["s"] = score(d)
    best = d.loc[d.groupby(["trip_id", "pt"]).s.idxmin()]
    return best.label.sum() / (PTS[PTS.partition == part].label_status == 0).sum()
RULES = {"nearest arc": lambda d: d.dist,
         "nearest arc pointing the vehicle's way": lambda d: d.dist + 1000 * ((d.heading_known == 1) & (d.cos_heading < 0)),
         "smallest ±1 s window distance": lambda d: d.win_dist_mean,
         "±1 s window distance, pointing the vehicle's way": lambda d: d.win_dist_mean + 1000 * ((d.heading_known == 1) & (d.cos_heading < 0))}
BASE = pd.DataFrame([{"rule": k, "train_top1": top1(FL, fn, "train"), "student_test_top1": top1(FL, fn, "student_test")} for k, fn in RULES.items()])
BASE.to_csv(f"{TAB}/12_rule_baselines.csv", index=False); display(BASE.round(4))
fig, ax = plt.subplots(figsize=(9, 3.5))
y = np.arange(len(BASE)); ax.barh(y - .2, BASE.train_top1, .4, label="train"); ax.barh(y + .2, BASE.student_test_top1, .4, label="student-test")
ax.set_yticks(y); ax.set_yticklabels(BASE.rule, fontsize=8); ax.set(xlim=(0, 1), xlabel="point-level Top-1 accuracy", title="Rule-based baselines")
ax.legend(); plt.tight_layout(); plt.savefig(f"{FIG}/04_rule_baselines.png", dpi=130); plt.show()

## 8. Folium maps of the derived labels
Layers: GPS points coloured by label status (green = labelled, red = unlabelled), the reference arc each labelled
point was matched to (thin blue line from the point to its projection), and the road network nearby.
Trip 11 = out-and-back (direction matters), trip 66 = local reference error, trip 54 = parked off-network.

In [ ]:
import folium
from pathsnap.geo import seg_project as _sp

def label_map(part, tid, max_pts=2500):
    raw = read_trip(trip_dir(DATA_ROOT, part, tid)); tr = raw.track
    lon0, lat0 = tr.lon.mean(), tr.lat.mean(); to_xy = make_projection(lon0, lat0); P = to_xy(tr.lon.values, tr.lat.values)
    net = build_network(raw.nodes, raw.arcs, lon0, lat0, aoi_xy=P)
    g = PTS[(PTS.partition == part) & (PTS.trip_id == tid)].set_index("pt")
    inv = lambda xy: (xy[:, 1] / C.R_EARTH * 180 / np.pi + lat0,
                      xy[:, 0] / (C.R_EARTH * np.cos(np.radians(lat0))) * 180 / np.pi + lon0)
    m = folium.Map(location=[tr.lat.mean(), tr.lon.mean()], zoom_start=14, tiles="cartodbpositron", control_scale=True)
    nd = raw.nodes
    fg_r = folium.FeatureGroup(name="Reference route (green)")
    for a in raw.route:
        u, v = raw.arcs[a]
        folium.PolyLine([[nd.lat[u], nd.lon[u]], [nd.lat[v], nd.lon[v]]], color="green", weight=4, opacity=.5).add_to(fg_r)
    fg_r.add_to(m)
    idx = np.linspace(0, len(P) - 1, min(max_pts, len(P))).astype(int)
    fg_l = folium.FeatureGroup(name="Point → matched reference arc (blue)")
    fg_ok = folium.FeatureGroup(name="Labelled points (green)"); fg_bad = folium.FeatureGroup(name="Unlabelled points (red)")
    for i in idx:
        st = int(g.label_status[i]); col = "#2ca02c" if st == 0 else "#d62728"
        folium.CircleMarker([tr.lat[i], tr.lon[i]], radius=2.5, color=col, fill=True, fill_opacity=.9, weight=0,
                            popup=f"pt {i}: {STATUS[st]}, ref arc {int(g.ref_arc[i])}, {g.ref_dist[i]:.1f} m").add_to(fg_ok if st == 0 else fg_bad)
        if st == 0:
            a = int(g.ref_arc[i]); A_, B_ = net.nxy[net.arcs[a, 0]], net.nxy[net.arcs[a, 1]]
            _, tt = _sp(P[i], A_, B_); q = (A_ + tt * (B_ - A_))[None]; la, lo = inv(q)
            folium.PolyLine([[tr.lat[i], tr.lon[i]], [float(la[0]), float(lo[0])]], color="#1f77b4", weight=1, opacity=.7).add_to(fg_l)
    for fg in (fg_l, fg_ok, fg_bad): fg.add_to(m)
    m.fit_bounds([[tr.lat.min(), tr.lon.min()], [tr.lat.max(), tr.lon.max()]])
    folium.LayerControl(collapsed=False).add_to(m)
    n_un = int((g.label_status != 0).sum())
    legend = f"""<div style="position:fixed;bottom:25px;left:25px;z-index:9999;background:white;padding:8px 10px;
      border:1px solid #999;border-radius:6px;font-size:13px"><b>{part} trip {tid} — derived labels</b><br>
      <span style="color:green">&#9473;&#9473;</span> reference route<br>
      <span style="color:#2ca02c">&#9679;</span> labelled point ({len(g)-n_un})<br>
      <span style="color:#d62728">&#9679;</span> unlabelled point ({n_un})<br>
      <span style="color:#1f77b4">&#9472;</span> point → matched arc</div>"""
    m.get_root().html.add_child(folium.Element(legend))
    return m

for part, tid in [("train", 11), ("train", 66), ("train", 54), ("student_test", 0)]:
    label_map(part, tid).save(f"{MAPS}/labels_trip_{tid:08d}_{part}.html"); print("saved label map", part, tid)
label_map("train", 11)

## 9. Save the dataset for M4 (+ data dictionary)
One row per (trip, GPS point, candidate arc). Keys: `partition, trip_id, pt, arc`. Model inputs: the 16 `FEATURES`.
Target: `label` (1 / 0; −1 = unlabelled point → excluded). `cv_group` = trip id, with trip 16 merged into 10 (D12)
→ use it for GroupKFold in M4/M6.

In [ ]:
def save(df, name):
    try:
        df.to_parquet(f"{DS}/{name}.parquet", index=False); return f"{name}.parquet"
    except Exception:                       # no pyarrow → pickle
        df.to_pickle(f"{DS}/{name}.pkl"); return f"{name}.pkl"
f1 = save(FEAT, "candidates_features"); f2 = save(PTS, "points")
meta = {"pathsnap_version": pathsnap.__version__, "features": FEATURES, "target": "label",
        "label_values": {"1": "candidate is the reference arc", "0": "candidate is not", "-1": "point unlabelled — exclude"},
        "group_column": "cv_group", "config": {k: getattr(C, k) for k in dir(C) if k.isupper()},
        "rows": int(len(FEAT)), "points": int(len(PTS)),
        "files": {"candidates": f1, "points": f2}}
json.dump(meta, open(f"{DS}/dataset_meta.json", "w"), indent=1, default=str)
print("saved", f1, f2, "and dataset_meta.json to", DS)

summary = f"""# M3 summary (generated by 02_preprocessing_features)
- Trips: {TIM.partition.value_counts().to_dict()}; GPS points {len(PTS):,}; candidate rows {len(FEAT):,}; processing {TIM.seconds.sum():.0f}s total.
- Labelled points: train {(PTS[PTS.partition=='train'].label_status==0).mean():.4f}, student-test {(PTS[PTS.partition=='student_test'].label_status==0).mean():.4f}.
- Unlabelled from trips 54/33/2: {unl.trip_id.isin([54,33,2]).mean():.3f} of all unlabelled.
- Alignment robustness: same reference arc under every setting change ≥ {SENS.same_arc_as_base.min():.4f}.
- Naive (nearest route arc) labels would differ on {100*NAIVE.naive_differs.sum()/NAIVE.labelled.sum():.2f}% of labelled train points (trip 11: {NAIVE.set_index('trip_id').loc[11,'naive_differs_share']:.2f}).
- Candidate miss rate (labelled): {cm.candidate_miss_rate.round(4).to_dict()}; mean candidates {cm.mean_candidates.round(2).to_dict()}.
- Positive share of rows: {cb.positive_share.round(4).to_dict()}.
- Strongest single features (train): {', '.join(AUC.feature.head(5))}.
- Rule baselines student-test Top-1: {dict(zip(BASE.rule, BASE.student_test_top1.round(4)))}.
"""
open(f"{OUT_DIR}/m3_summary.md", "w").write(summary); print(summary)